# Search Strategies — Part 3

**One fact. Two questions. Every search.**

Same answer, same documents — only the search changes. Watch the output change with it.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]  # docs/videos/ -> repo root
sys.path.insert(0, str(ROOT))

from rag.search import kb_search_client  # .env is loaded automatically via rag.db

EMPRESA = "Empresa X"
CHUNKING = "semantic"

# From tests/rag/rag_queries.yaml — we already know the answer.
ANSWER = "99.9%"
SOURCE_DOC = "servico_b.md"

Q_EXACT      = "What is the SLA for Service B?"            # uses the document's own words
Q_PARAPHRASE = "How reliable is Service B supposed to be?"  # same question, different words

In [2]:
from html import escape
from IPython.display import HTML, display

from rag.ingest import get_embedder
from rag.search import query_candidates

emb = get_embedder()

TOP = 5
STYLE = """
<style>
  .s { border-collapse: collapse; font: 14px/1.4 system-ui, sans-serif; color: #1a1a1a; background: #fff; }
  .s th, .s td { border: 1px solid #ddd; padding: 8px 10px; vertical-align: top; text-align: left; }
  .s thead th { background: #f2f2f2; }
  .s .doc { font: 12px ui-monospace, monospace; color: #666; }
  .s .hit  { background: #d7f5dc; }
  .s .near { background: #fff3b0; }
  .s .txt  { background: #dbeafe; }
  .s .none { color: #b00020; font-style: italic; }
</style>
"""


def search(query: str, **settings) -> list[dict]:
    return kb_search_client.invoke({
        "query": query, "k": TOP, "empresa": EMPRESA, "chunking": CHUNKING,
        "reranker": "none", **settings,
    })


def verdict(r: dict) -> str:
    """hit = the source document; near = right number, but from another document."""
    if Path(r["doc_path"]).name == SOURCE_DOC:
        return "hit"
    return "near" if ANSWER in (r.get("content") or "") else ""


def column(results) -> str:
    if isinstance(results, Exception):
        return f'<td class="none">error: {escape(str(results))[:120]}</td>'
    if not results:
        return '<td class="none">(nothing came back)</td>'
    rows = []
    for i, r in enumerate(results, 1):
        text = " ".join((r.get("content") or "").split())[:140]
        rows.append(f'<div class="{verdict(r)}" style="padding:4px;margin:2px 0">'
                    f'<b>#{i}</b> <span class="doc">{escape(Path(r["doc_path"]).name)} · score {r["score"]:.3f}</span><br>'
                    f'{escape(text)}…</div>')
    return f"<td>{''.join(rows)}</td>"


def run_safe(query, **settings):
    try:
        return search(query, **settings)
    except Exception as e:  # e.g. DB unreachable — show it in the table instead of crashing
        return e


def both(**settings) -> None:
    """Run both questions with the same settings and show them side by side."""
    display(HTML(STYLE + f"""
    <p class="doc">settings: {escape(str(settings))}</p>
    <table class="s"><thead><tr>
      <th style="width:50%">{escape(Q_EXACT)}</th><th style="width:50%">{escape(Q_PARAPHRASE)}</th>
    </tr></thead><tbody><tr>
      {column(run_safe(Q_EXACT, **settings))}{column(run_safe(Q_PARAPHRASE, **settings))}
    </tr></tbody></table>
    <p class="doc">green = source document · yellow = right number, other document</p>
    """))

def table(headers: list[str], rows: list[list], classes: list[str] | None = None) -> None:
    """Render a plain table; `classes` gives each row a CSS class (e.g. "hit")."""
    classes = classes or [""] * len(rows)
    head = "".join(f"<th>{escape(str(h))}</th>" for h in headers)
    body = "".join(
        f'<tr class="{c}">' + "".join(f"<td>{escape(str(v))}</td>" for v in row) + "</tr>"
        for row, c in zip(rows, classes)
    )
    display(HTML(STYLE + f'<table class="s"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))


def name(r: dict) -> str:
    return f'{Path(r["doc_path"]).name}#{r["chunk_ix"]}'


def row_class(r: dict) -> str:
    return "hit" if Path(r["doc_path"]).name == SOURCE_DOC else ""


## Step 0 — the answer

Before any search: this is the line we want back.

In [3]:
doc = ROOT / "kb/empresa_x/services" / SOURCE_DOC
for line in doc.read_text().splitlines():
    if ANSWER in line:
        print(f"{SOURCE_DOC}:  {line.strip()}")

servico_b.md:  - SLA: 99.9% monthly uptime.


## 1. Text search

Ctrl+F: finds chunks that contain **every** word of the question.

**Predict:** question 1 uses the document's words, question 2 doesn't. What comes back for each?

In [4]:
both(search_type="text")

What is the SLA for Service B?,How reliable is Service B supposed to be?
"#1 lead_maria.md · score 0.178# Lead Report — Maria (Empresa X) ## Executive Summary - Industry: agricultural-input retail (mid-size) - Estimated budget: R$ 80,000 - Desi…#2 servico_b.md · score 0.139# Service B — SLA and Scope (Empresa X) ## Description Service B consists of monitoring, preventive maintenance, and N2 support for incident…#3 lead_joao.md · score 0.126# Lead Report — João (Empresa X) ## Executive Summary - Industry: agriculture (soybean farming) — agro - Size: mid-large - Estimated budget:…",(nothing came back)


## 2. Vector search

Finds chunks with the closest **meaning**, whatever the words.

**Predict:** does question 2 work now? Does question 1 still work?

In [5]:
both(search_type="vector")

What is the SLA for Service B?,How reliable is Service B supposed to be?
"#1 servico_b.md · score 0.677# Service B — SLA and Scope (Empresa X) ## Description Service B consists of monitoring, preventive maintenance, and N2 support for incident…#2 produto_a.md · score 0.421# Product A — Specifications and Policies (Empresa X) ## Overview Product A was built for mid-size operations, focused on stability, ease of…#3 cancelamento.md · score 0.406# Cancellation Policy (Empresa X) ## Scope Applies to Product A and Service B contracts. ## Cooling-Off Period - Cancellation allowed within…#4 lead_maria.md · score 0.383# Lead Report — Maria (Empresa X) ## Executive Summary - Industry: agricultural-input retail (mid-size) - Estimated budget: R$ 80,000 - Desi…#5 lead_joao.md · score 0.316# Lead Report — João (Empresa X) ## Executive Summary - Industry: agriculture (soybean farming) — agro - Size: mid-large - Estimated budget:…","#1 servico_b.md · score 0.526# Service B — SLA and Scope (Empresa X) ## Description Service B consists of monitoring, preventive maintenance, and N2 support for incident…#2 produto_a.md · score 0.334# Product A — Specifications and Policies (Empresa X) ## Overview Product A was built for mid-size operations, focused on stability, ease of…#3 lead_maria.md · score 0.318# Lead Report — Maria (Empresa X) ## Executive Summary - Industry: agricultural-input retail (mid-size) - Estimated budget: R$ 80,000 - Desi…#4 cancelamento.md · score 0.271# Cancellation Policy (Empresa X) ## Scope Applies to Product A and Service B contracts. ## Cooling-Off Period - Cancellation allowed within…#5 lead_joao.md · score 0.266# Lead Report — João (Empresa X) ## Executive Summary - Industry: agriculture (soybean farming) — agro - Size: mid-large - Estimated budget:…"


## 3. Hybrid — the process

Text and vector each fail somewhere. Hybrid runs **both** and merges the two lists.

**Step A:** the two lists, for the exact question.

In [6]:
q = Q_EXACT
vec = query_candidates(q, 24, "vector", None, EMPRESA, CHUNKING, emb.embed_query(q), None)
txt = query_candidates(q, 24, "text",   None, EMPRESA, CHUNKING, None, None)

print("VECTOR list")
table(["rank", "chunk", "score"], [[i, name(r), f'{r["score"]:.3f}'] for i, r in enumerate(vec, 1)],
      [row_class(r) for r in vec])
print("TEXT list")
table(["rank", "chunk", "score"], [[i, name(r), f'{r["score"]:.3f}'] for i, r in enumerate(txt, 1)],
      [row_class(r) for r in txt])

VECTOR list


rank,chunk,score
1,servico_b.md#0,0.677
2,produto_a.md#0,0.421
3,cancelamento.md#0,0.406
4,lead_maria.md#0,0.383
5,lead_joao.md#0,0.316


TEXT list


rank,chunk,score
1,lead_maria.md#0,0.178
2,servico_b.md#0,0.139
3,lead_joao.md#0,0.126


The two score columns don't mean the same thing: vector is a similarity (0–1), text is a
keyword-match rank (`ts_rank_cd`). **How do you merge them?** Two ways.

## 4. Merge by score — union

Put both lists together and sort by score.

**Predict:** where do the text results end up?

In [7]:
merged = [("vector", r) for r in vec] + [("text", r) for r in txt]
merged.sort(key=lambda x: x[1]["score"], reverse=True)
table(["#", "from", "chunk", "score"],
      [[i, src, name(r), f'{r["score"]:.3f}'] for i, (src, r) in enumerate(merged, 1)],
      ["hit" if row_class(r) else ("txt" if src == "text" else "") for src, r in merged])

#,from,chunk,score
1,vector,servico_b.md#0,0.677
2,vector,produto_a.md#0,0.421
3,vector,cancelamento.md#0,0.406
4,vector,lead_maria.md#0,0.383
5,vector,lead_joao.md#0,0.316
6,text,lead_maria.md#0,0.178
7,text,servico_b.md#0,0.139
8,text,lead_joao.md#0,0.126


Blue = came from text. Here every text score is lower than every vector score, so text never
changes anything — union is just vector search with leftovers.

## 5. Merge by rank — RRF

Ignore the scores. Use only each chunk's **position** in each list:

`score = 1.5 / (60 + vector_rank) + 1.0 / (60 + text_rank)`

A chunk high in both lists gets both halves. **Predict:** which chunk wins?

In [8]:
v_rank = {name(r): i for i, r in enumerate(vec, 1)}
t_rank = {name(r): i for i, r in enumerate(txt, 1)}
by_name = {name(r): r for r in vec + txt}

rrf_rows = []
for n in by_name:
    v = 1.5 / (60 + v_rank[n]) if n in v_rank else 0.0
    t = 1.0 / (60 + t_rank[n]) if n in t_rank else 0.0
    rrf_rows.append((n, v_rank.get(n, "—"), t_rank.get(n, "—"), v, t, v + t))
rrf_rows.sort(key=lambda x: x[-1], reverse=True)

table(["chunk", "vector rank", "text rank", "1.5/(60+v)", "1.0/(60+t)", "RRF score"],
      [[n, vr, tr, f"{v:.4f}", f"{t:.4f}", f"{s:.4f}"] for n, vr, tr, v, t, s in rrf_rows],
      [row_class(by_name[r[0]]) for r in rrf_rows])

chunk,vector rank,text rank,1.5/(60+v),1.0/(60+t),RRF score
servico_b.md#0,1,2,0.0246,0.0161,0.0407
lead_maria.md#0,4,1,0.0234,0.0164,0.0398
lead_joao.md#0,5,3,0.0231,0.0159,0.0389
produto_a.md#0,2,—,0.0242,0.0000,0.0242
cancelamento.md#0,3,—,0.0238,0.0000,0.0238


## 6. Add-on: HyDE — the process

The paraphrase scored lower than the exact question. HyDE's idea: don't search with the
question — ask an LLM to **imagine the answer**, and search with that. An imagined answer
sounds like a document, so it lands closer to the real one.

**Step A:** the LLM writes a fake answer. It has never seen our documents.

In [9]:
from rag.search import hyde

hyde_text = hyde(Q_PARAPHRASE)
print("QUESTION:\n ", Q_PARAPHRASE)
print("\nWHAT WE SEARCH WITH INSTEAD (invented by the LLM):\n ", hyde_text)

QUESTION:
  How reliable is Service B supposed to be?

WHAT WE SEARCH WITH INSTEAD (invented by the LLM):
  Service B is designed to achieve a high reliability target of 99.9% uptime, corresponding to less than 8.76 hours of downtime annually. This reliability level is supported by its redundant architecture, automated failover mechanisms, and continuous health monitoring. Additionally, Service B undergoes rigorous stress testing and is backed by a comprehensive incident response plan to minimize service interruptions. Service Level Agreements (SLAs) reflect these commitments, ensuring that any deviations are promptly addressed with corrective actions to maintain consistent operational availability.


**Step B:** search with each one and compare the scores for every chunk.

**Predict:** does the right chunk score higher with the fake answer than with the question?

In [10]:
def scores_for(text: str) -> dict:
    found = query_candidates(text, 24, "vector", None, EMPRESA, CHUNKING, emb.embed_query(text), None)
    return {name(r): (r, r["score"]) for r in found}


before, after = scores_for(Q_PARAPHRASE), scores_for(hyde_text)
order = sorted(before, key=lambda n: before[n][1], reverse=True)
table(["chunk", "score with question", "score with HyDE text", "change"],
      [[n, f"{before[n][1]:.3f}", f"{after[n][1]:.3f}", f"{after[n][1] - before[n][1]:+.3f}"] for n in order],
      [row_class(before[n][0]) for n in order])

chunk,score with question,score with HyDE text,change
servico_b.md#0,0.526,0.676,+0.150
produto_a.md#0,0.334,0.436,+0.101
lead_maria.md#0,0.318,0.401,+0.084
cancelamento.md#0,0.271,0.328,+0.058
lead_joao.md#0,0.266,0.341,+0.074


**Cost:** one extra LLM call before every search, and the fake answer changes each run.

## 7. Add-on: rerank — the process

Search compares the question and each chunk **separately** (two vectors). A reranker reads
the question and a chunk **together**, like a person would — slower, so it only re-reads
the candidates search already found.

**Step A:** the RRF order from above — look how close the top scores are.

In [11]:
candidates = [by_name[r[0]] for r in rrf_rows]
table(["rank", "chunk", "RRF score"], [[i, r[0], f"{r[-1]:.4f}"] for i, r in enumerate(rrf_rows, 1)],
      [row_class(c) for c in candidates])

rank,chunk,RRF score
1,servico_b.md#0,0.0407
2,lead_maria.md#0,0.0398
3,lead_joao.md#0,0.0389
4,produto_a.md#0,0.0242
5,cancelamento.md#0,0.0238


**Step B:** the reranker gives each (question, chunk) pair a relevance score from 0 to 1.

**Predict:** is the reranker as unsure as RRF was?

In [12]:
from langchain_cohere import CohereRerank  # the same model rag/search.py:apply_rerank uses

scores = CohereRerank(model="rerank-english-v3.0").rerank(
    query=q, documents=[c["content"] for c in candidates], top_n=len(candidates)
)
out = []
for new, s in enumerate(scores, 1):
    old = s["index"] + 1
    move = "=" if old == new else (f"▲ {old - new}" if old > new else f"▼ {new - old}")
    out.append(([f"#{old}", f"#{new}", move, name(candidates[s["index"]]), f'{s["relevance_score"]:.4f}'],
                row_class(candidates[s["index"]])))
table(["before", "after", "move", "chunk", "relevance"], [r for r, _ in out], [c for _, c in out])

before,after,move,chunk,relevance
#1,#1,=,servico_b.md#0,0.9999
#3,#2,▲ 1,lead_joao.md#0,0.9861
#2,#3,▼ 1,lead_maria.md#0,0.9736
#5,#4,▲ 1,cancelamento.md#0,0.0883
#4,#5,▼ 1,produto_a.md#0,0.0000


Same chunks, new order — rerank can't find anything search missed. It only fixes the order.

## So which one is right?

Same fact, six settings, different outputs — and two questions is not proof. `eval/run.py` runs
every setting over **all** of `tests/rag/rag_queries.yaml` and scores it.

`make rag_phase2_save_fast`